In [1]:
import pandas as pd
import numpy as np
# from Bio import SeqIO

import gc
gc.collect()
# np.set_printoptions(threshold=np.inf)
# pd.set_option('display.max_rows', 10)

0

In [2]:
def find_name_collisions(df, levels):
    collisions = {}

    for i in range(1, len(levels)):  # skip root level (phylum has no parent)
        child = levels[i]
        parents = levels[:i]

        # Count unique parent combinations per child name
        grouped = (
            df.groupby(child)[parents]
            .nunique()
        )

        # A collision occurs if ANY parent column has >1 unique value
        mask = (grouped > 1).any(axis=1)

        collision_names = grouped[mask].index.tolist()

        if collision_names:
            collisions[child] = collision_names

    return collisions

# Data Insights

## Load Data

In [2]:
def fasta_to_df(file_path):
    # 1. Define the levels we expect to find
    levels = {'k__': 'kingdom', 'p__': 'phylum', 'c__': 'class', 
              'o__': 'order', 'f__': 'family', 'g__': 'genus', 's__': 'species'}
    
    data = []
    for record in SeqIO.parse(file_path, "fasta"):
        # prefill with NA
        row = {name: pd.NA for name in levels.values()}
        
        # Split header: Accession|Taxonomy|ID
        parts = record.description.split('|')
        # row['accession'] = parts[0]
        row['dna_barcode'] = str(record.seq)
        
        # Parse taxonomy string if it exists
        if len(parts) > 1:
            taxa_parts = parts[1].split(';')
            for item in taxa_parts:
                prefix = item[:3] # Get 'k__', 'p__', etc.
                if prefix in levels:
                    row[levels[prefix]] = item[3:].replace('_', ' ')
        
        data.append(row)
    
    return pd.DataFrame(data)

## General Insights

In [5]:
# df = fasta_to_df("../datasets/mycoai/test3.fasta")
df = pd.read_csv("../datasets/mycoai_full/train_labels.csv")
# df = df.sample(n=50000, random_state=42)
# df.to_csv("train_50K.csv", index=False)

In [6]:
df

,root_id,kingdom_id,phylum_id,class_id,order_id,family_id,genus_id,species_id
0,0,1,14,19,195,713,2687,11796
1,0,1,14,31,111,401,1371,5343
2,0,1,12,41,216,826,3211,16027
3,0,1,14,19,195,712,2709,12293
4,0,1,12,41,233,871,3549,17883
...,...,...,...,...,...,...,...,...
354690,0,1,14,26,143,523,1816,7830
354691,0,1,14,19,205,702,2628,11541
354692,0,1,12,41,215,839,3405,17173
354693,0,1,12,46,275,1020,4065,20568


In [ ]:
df.describe(include='all')

,phylum,class,order,family,genus,species,dna_barcode
count,11619,11619,11619,11619,11619,11619,11619
unique,10,41,149,448,1667,5537,10721
top,Ascomycota,Sordariomycetes,Hypocreales,Aspergillaceae,Penicillium,Exophiala dermatitidis,AACAAGGTTTCCGTTGGTGAACCAGCGGAGGGATCATTACCGAGTT...
freq,9431,4041,1768,1516,735,98,24


In [ ]:
# df = df.sample(n=50000, random_state=42)
# df.describe(include='all')

,kingdom,phylum,class,order,family,genus,species,dna_barcode
count,50000,50000,50000,50000,50000,50000,50000,50000
unique,1,12,53,169,556,2226,10468,49921
top,Fungi,Ascomycota,Sordariomycetes,Hypocreales,Aspergillaceae,Fusarium,Fusarium oxysporum,TGGAAGTAAAAGTCGTAACAAGGTCTCCGTAGGTGAACCTGCGGAG...
freq,50000,33928,13031,7272,4206,2851,761,3


In [ ]:
# Make sure kingdom is a single value so it's safe to remove to match BioScan

# print("Kingdoms", df['kingdom'].unique())
print("max_length", df['dna_barcode'].str.len().max())
print("has Na: ", df.isna().any().any())    # We have entries that are rather '?' # 1062 all train - 1251 test1 - 2501 test2

max_length 1251
has Na:  False


: 

## Count of Sequence Lengths

In [121]:
df['seq_length'] = df['dna_barcode'].str.len()
length_counts = df['seq_length'].value_counts().sort_index()

print("Length | Count")
print("-" * 15)
for length, count in length_counts.items():
    print(f"{length:6} | {count}")

Length | Count
---------------
   281 | 1
   284 | 1
   285 | 1
   286 | 1
   290 | 1
   293 | 2
   297 | 2
   299 | 5
   301 | 1
   306 | 3
   307 | 4
   308 | 2
   309 | 5
   310 | 2
   311 | 18
   312 | 46
   313 | 83
   314 | 134
   315 | 142
   316 | 67
   317 | 42
   318 | 95
   319 | 47
   320 | 80
   321 | 37
   322 | 29
   323 | 41
   324 | 38
   325 | 23
   326 | 25
   327 | 13
   328 | 20
   329 | 33
   330 | 46
   331 | 50
   332 | 29
   333 | 23
   334 | 20
   335 | 14
   336 | 32
   337 | 17
   338 | 25
   339 | 7
   340 | 6
   341 | 3
   342 | 4
   343 | 6
   344 | 2
   345 | 2
   346 | 2
   347 | 10
   348 | 9
   349 | 19
   350 | 8
   351 | 5
   352 | 6
   353 | 4
   354 | 3
   355 | 2
   357 | 2
   358 | 1
   359 | 1
   360 | 1
   361 | 1
   362 | 1
   363 | 4
   364 | 6
   365 | 3
   368 | 2
   370 | 2
   371 | 2
   372 | 1
   373 | 3
   374 | 5
   375 | 9
   376 | 16
   377 | 4
   378 | 6
   379 | 15
   380 | 17
   381 | 4
   382 | 8
   383 | 17
   384 | 17
   385 |

## The Question Marks

In [20]:
# df["class"].unique()
# df = df.replace('?', np.nan)

In [21]:
# file_path = "../datasets/mycoai/test3.fasta"
# stop = 32
# i = 0
# for record in SeqIO.parse(file_path, "fasta"):
#     i += 1
#     if i > stop:
#         break
# record.description


# sample number 32 is:
# '|k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__?;g__Candida;s__Candida_albicans|SH0987662.09FU'

In [52]:
mask = (df == '?').any(axis=1)
rows_with_question_mark = df[mask]
print(f"Found {len(rows_with_question_mark)} rows with '?'")
rows_with_question_mark

Found 0 rows with '?'


,phylum,class,order,family,genus,species,dna_barcode,seq_length


In [23]:
def remove_question_marks(df):
    initial_count = len(df)
    rows_with_question_mark = df.isin(['?']).any(axis=1)
    df = df[~rows_with_question_mark]
    removed_count = initial_count - len(df)
    print(f"Removed {removed_count} rows containing '?'")
    return df

df = remove_question_marks(df)

Removed 12642 rows containing '?'


## Barcode Sequences

In [53]:
# 1. Join all sequences into one string, then get unique characters
unique_letters = set[str]("".join(df['dna_barcode']))

# 2. Print the result sorted alphabetically
print(f"Unique characters found: {sorted(unique_letters)}")

Unique characters found: ['A', 'B', 'C', 'D', 'G', 'H', 'K', 'M', 'N', 'R', 'S', 'T', 'V', 'W', 'Y']


In [28]:
def standardize_barcode(df):
    df['dna_barcode'] = df['dna_barcode'].str.replace(r'[^ACGT]', 'N', regex=True, case=False)
    df['dna_barcode'] = df['dna_barcode'].str.upper()

print("Conversion complete. All non-canonical bases are now 'N'.")

Conversion complete. All non-canonical bases are now 'N'.


# Create Test Data

In [89]:
def check_overlap(df_small, df_big):
    df_small = df_small[["dna_barcode"]]
    df_big = df_big[["dna_barcode"]]

    check_merge = df_small.merge(df_big, how='left', indicator=True)
    all_unique = (check_merge['_merge'] == 'left_only').all()
    print(f"Zero overlap (Every row is unseen): {all_unique}")

def check_merge(df_small, df_big):
    df_small = df_small.drop(columns=["dna_barcode"]).drop_duplicates()
    df_big = df_big.drop(columns=["dna_barcode"]).drop_duplicates()

    check_merge = df_small.merge(df_big, how='left', indicator=True)
    is_subset = (check_merge['_merge'] == 'both').all()
    print(f"Every taxon exists in the train: {is_subset}")

def create_test_data(full_df, train_df):
    taxa_cols = ['phylum', 'class', 'order', 'family', 'genus', 'species']
    unique_taxa = train_df[taxa_cols].drop_duplicates()

    test_df = full_df.merge(unique_taxa, on=taxa_cols)

    test_df = test_df.merge(
        train_df[['dna_barcode']].drop_duplicates(), 
        on='dna_barcode', 
        how='left', 
        indicator=True
    )

    test_df = test_df[test_df['_merge'] == 'left_only'].drop(columns=['_merge'])
    return test_df

def filter_test_data(test_df, trainset):
    taxa_cols = ['phylum', 'class', 'order', 'family', 'genus', 'species']
    unique_taxa = trainset[taxa_cols].drop_duplicates()

    return test_df.merge(unique_taxa, on=taxa_cols, how='inner')

def map_barcode_indices(small_df, big_df, barcode_col='dna_barcode'):
    big_lookup = {barcode: idx for idx, barcode in enumerate(big_df[barcode_col])}
    small_to_big_map = {}
    
    for small_idx, barcode in enumerate(small_df[barcode_col]):
        if barcode in big_lookup:
            # Store the mapping: {Small Index: Big Index}
            small_to_big_map[small_idx] = big_lookup[barcode]
            
    return small_to_big_map

In [90]:
trainset = pd.read_csv("../datasets/mycoai/trainset_clean.csv")
test_df = pd.read_csv("../datasets/mycoai/test_1/test1.csv")
# df_50k = pd.read_csv("../datasets/mycoai/train_50K.csv")

In [91]:
test_df

,phylum,class,order,family,genus,species,dna_barcode
0,Ascomycota,Saccharomycetes,Saccharomycetales,Saccharomycetaceae,Zygotorulaspora,Zygotorulaspora mrakii,GTAACAAGGTTTCCGTAGGTGAACCTGCGGAAGGATCATTATAGAA...
1,Ascomycota,Saccharomycetes,Saccharomycetales,Saccharomycetaceae,Zygotorulaspora,Zygotorulaspora mrakii,AACAAGGTTTCCGTAGGTGAACCTGCGGAAGGATCATTATAGAAAA...
2,Ascomycota,Saccharomycetes,Saccharomycetales,Saccharomycetaceae,Zygotorulaspora,Zygotorulaspora florentina,ATGCCGCCTTAACTTGCGCTGACAACATACACACAGTGGAGATATA...
3,Ascomycota,Saccharomycetes,Saccharomycetales,Saccharomycetaceae,Zygotorulaspora,Zygotorulaspora florentina,AGGTGAACCTGCGGAAGGATCATTACTGAAAGTCATGCGCTTAACT...
4,Ascomycota,Saccharomycetes,Saccharomycetales,Saccharomycetaceae,Zygotorulaspora,Zygotorulaspora florentina,AAGTTTCCGTAGTGAACCTGCGGAAGGATCATTACTGAAAGTCATG...
...,...,...,...,...,...,...,...
4434,Ascomycota,Saccharomycetes,Saccharomycetales,Saccharomycopsidaceae,Ambrosiozyma,Ambrosiozyma angophorae,GTAACAAGGTTTCCGTAGGTGAACCTGCGGAAGGATCATTACAGTA...
4435,Ascomycota,Saccharomycetes,Saccharomycetales,Saccharomycopsidaceae,Ambrosiozyma,Ambrosiozyma angophorae,CGTAACAAGGTTTCCGTAGGTGAACCTGCGGAAGGATCATTACAGT...
4436,Ascomycota,Saccharomycetes,Saccharomycetales,Saccharomycopsidaceae,Ambrosiozyma,Ambrosiozyma angophorae,AATACGATTGATGGCTTAGTGAGGCTTCAGGATTAGTTTAGAGAAG...
4437,Ascomycota,Saccharomycetes,Saccharomycetales,Saccharomycopsidaceae,Ambrosiozyma,Ambrosiozyma platypodis,GTTTCCGTAGGTGAACCTGCGGAAGGATCATTACAGTATCTTTCTA...


In [92]:
test_df = filter_test_data(test_df, trainset)
test_df

,phylum,class,order,family,genus,species,dna_barcode
0,Ascomycota,Saccharomycetes,Saccharomycetales,Saccharomycetaceae,Zygotorulaspora,Zygotorulaspora florentina,ATGCCGCCTTAACTTGCGCTGACAACATACACACAGTGGAGATATA...
1,Ascomycota,Saccharomycetes,Saccharomycetales,Saccharomycetaceae,Zygotorulaspora,Zygotorulaspora florentina,AGGTGAACCTGCGGAAGGATCATTACTGAAAGTCATGCGCTTAACT...
2,Ascomycota,Saccharomycetes,Saccharomycetales,Saccharomycetaceae,Zygotorulaspora,Zygotorulaspora florentina,AAGTTTCCGTAGTGAACCTGCGGAAGGATCATTACTGAAAGTCATG...
3,Ascomycota,Saccharomycetes,Saccharomycetales,Saccharomycetaceae,Zygotorulaspora,Zygotorulaspora florentina,TGCGGGAAGGATCATTACTGAAAGTCATGCCGCCTTAACTGCGCTG...
4,Ascomycota,Saccharomycetes,Saccharomycetales,Saccharomycetaceae,Zygotorulaspora,Zygotorulaspora florentina,GTCGTAACAAGGTTTCCGTAGGTGAACCTGCGGAAGGATCATTACT...
...,...,...,...,...,...,...,...
3116,Ascomycota,Saccharomycetes,Saccharomycetales,Saccharomycopsidaceae,Ambrosiozyma,Ambrosiozyma angophorae,GTAACAAGGTTTCCGTAGGTGAACCTGCGGAAGGATCATTACAGTA...
3117,Ascomycota,Saccharomycetes,Saccharomycetales,Saccharomycopsidaceae,Ambrosiozyma,Ambrosiozyma angophorae,CGTAACAAGGTTTCCGTAGGTGAACCTGCGGAAGGATCATTACAGT...
3118,Ascomycota,Saccharomycetes,Saccharomycetales,Saccharomycopsidaceae,Ambrosiozyma,Ambrosiozyma angophorae,AATACGATTGATGGCTTAGTGAGGCTTCAGGATTAGTTTAGAGAAG...
3119,Ascomycota,Saccharomycetes,Saccharomycetales,Saccharomycopsidaceae,Ambrosiozyma,Ambrosiozyma platypodis,GTTTCCGTAGGTGAACCTGCGGAAGGATCATTACAGTATCTTTCTA...


In [93]:
length_counts = test_df['dna_barcode'].str.len().value_counts().sort_index(ascending=True)

print(length_counts)

dna_barcode
304     1
317     1
323     2
325     1
332     1
       ..
1038    1
1052    1
1054    1
1063    1
1130    1
Name: count, Length: 541, dtype: int64


In [29]:
check_merge(test_df, trainset)

Every taxon exists in the train: True


In [95]:
loo_map = map_barcode_indices(test_df, trainset)

In [96]:
len(loo_map)

3115

In [ ]:
import pickle

# SAVE
with open('loo_map.pkl', 'wb') as f:
    pickle.dump(loo_map, f)

In [45]:
# sample_df.to_csv("test_10k.csv", index=False)
test_df.to_csv("test2_clean.csv", index=False)

# Out of Distribution

In [2]:
trainset = pd.read_csv("../datasets/mycoai_full/fasta/trainset.csv")
tree_df = pd.read_csv("../datasets/mycoai_full/trainset_clean.csv")
# df_50k = pd.read_csv("../datasets/mycoai/train_50K.csv")

In [3]:
def standardize_barcode(df):
    df['dna_barcode'] = df['dna_barcode'].str.replace(r'[^ACGT]', 'N', regex=True, case=False)
    df['dna_barcode'] = df['dna_barcode'].str.upper()
    
    return df

# To check same child name under different parents
cols = ['phylum', 'class', 'order', 'family', 'genus', 'species']

for i in range(1, len(cols)):
    parent = cols[i-1]
    child = cols[i]
    
    conflicts = tree_df.groupby(child)[parent].nunique()
    multi_parent = conflicts[conflicts > 1]
    
    if not multi_parent.empty:
        print(f"\nConflict found: The level '{child}' has names belonging to multiple '{parent}'s:")
        print(multi_parent.head())

## Missing

In [ ]:
# tree_copy = tree_df.copy()
# df_50k_copy = df_50k.copy()

# cols = ["phylum", "class", "order", "family", "genus", "species"]

# result = tree_copy.merge(
#     df_50k_copy[cols].drop_duplicates(),
#     on=cols,
#     how="left",
#     indicator=True
# )

# not_in_small = result[result["_merge"] == "left_only"].drop(columns=["_merge"])

In [5]:
cols_no_species = ["phylum", "class", "order", "family", "genus"]
cols_full = cols_no_species + ["species"]

match_genus = tree_df.merge(
    df_50k[cols_no_species].drop_duplicates(),
    on=cols_no_species,
    how="inner"
)

exact_matches = tree_df.merge(
    df_50k[cols_full].drop_duplicates(),
    on=cols_full,
    how="inner"
)


result = match_genus.merge(
    exact_matches[cols_full],
    on=cols_full,
    how="left",
    indicator=True
)

missing_species = result[result["_merge"] == "left_only"].drop(columns=["_merge"])

In [6]:
# Checks

overlap = set(missing_species["species"]) & set(df_50k["species"])
print(len(overlap))  # ideally 0 or very small

cols_no_species = ["phylum", "class", "order", "family", "genus"]

for col in cols_no_species:
    missing = set(result[col]) - set(df_50k[col])
    print(f"{col}: missing values = {len(missing)}")

0
phylum: missing values = 0
class: missing values = 0
order: missing values = 0
family: missing values = 0
genus: missing values = 0


In [7]:
missing_species

,phylum,class,order,family,genus,species,dna_barcode
2905,Basidiomycota,Agaricomycetes,Agaricales,Cortinariaceae,Cortinarius,Cortinarius caesiocanescens,GTGAACCTGCGGAGGATCATTATTGAATAAACCTGATAAGTTGCTG...
14709,Mucoromycota,Mucoromycetes,Mucorales,Mucoraceae,Mucor,Mucor laxorrhizus,CAGGATGATTTTAATCGAAGCCATGGTCAAGCCGACTTTTTTTCAG...
37945,Ascomycota,Eurotiomycetes,Onygenales,Onygenaceae,Chrysosporium,Chrysosporium vallenarense,TTCTTGTCTACTGACCCAGTTGCCTCGGTGGGCCGAGCCGTTCGCG...
39379,Ascomycota,Dothideomycetes,Pleosporales,Didymellaceae,Phoma,Phoma aloes,TAGCTTAATCGCGTGATGAGCAGCTGGTCTCTTTTCTCTACCCTTG...
44920,Basidiomycota,Agaricomycetes,Polyporales,Fomitopsidaceae,Resinoporia,Resinoporia cincta,TTGTAGCTGGCCTTTCTTCAGGCATGTGCACGCCCCGCTTCATCCA...
...,...,...,...,...,...,...,...
16869019,Basidiomycota,Exobasidiomycetes,Entylomatales,Entylomataceae,Entyloma,Entyloma hieracii,GGTGAACCTGCAGATGGATCATTAGTGAATAACAAGGGGGTTCCCA...
16869034,Basidiomycota,Agaricomycetes,Agaricales,Hygrophoraceae,Hygrophorus,Hygrophorus camarophyllus,AGAATTTTTTTTTGAANGGGTTTGTTGCTGGTCAAGTAAGACATGT...
16871405,Ascomycota,Sordariomycetes,Glomerellales,Glomerellaceae,Colletotrichum,Colletotrichum noveboracense,AAGGTCTCCGTTGGTGAACCAGCGGAGGGATCATTACTGAGTTTAC...
16871555,Ascomycota,Arthoniomycetes,Arthoniales,Roccellaceae,Dirina,Dirina mexicana,ATCAGAGACAGGGCCTCTTCAGAGCCCGACCTCCAACCCTCTGCCT...


In [8]:
missing_species = standardize_barcode(missing_species)
missing_species.to_csv("missing_species.csv", index=False)

## Unknown

In [4]:
trainset_filtered = trainset.drop(columns=["dna_barcode", "species", "genus"]).drop_duplicates()

mask = (trainset_filtered == "?").any(axis=1)

trainset_unknown = trainset_filtered[mask]
# trainset_known = trainset_filtered[~mask]

In [5]:
trainset_unknown

,phylum,class,order,family
15,?,?,?,?
27,Rozellomycota,?,?,?
33,Basidiomycota,Agaricomycetes,?,?
50,Ascomycota,Eurotiomycetes,Chaetothyriales,?
51,Basidiomycota,?,?,?
...,...,...,...,...
3445984,Ascomycota,Eurotiomycetes,Coryneliales,?
3507428,Ascomycota,Sordariomycetes,Melanosporales,?
3692018,Basidiomycota,Pucciniomycetes,Septobasidiales,?
3990974,Basidiomycota,Tremellomycetes,Holtermanniales,?


In [8]:
trainset_unknown.describe()

,phylum,class,order,family,genus
count,1098,1098,1098,1098,1098
unique,19,67,200,553,316
top,Ascomycota,Sordariomycetes,?,?,?
freq,674,209,166,520,783


In [6]:
# Define your 7 levels (adjust names to match your CSV)
# levels = ['phylum', 'class', 'order', 'family', 'genus', 'species']
# higher_taxa = ['phylum', 'class', 'order', 'family', 'genus']

levels = ['phylum', 'class', 'order', 'family']
higher_taxa = ['phylum', 'class', 'order']

# species_unknown_mask = (trainset_unknown['species'] == '?')
species_unknown_mask = (trainset_unknown['family'] == '?')

higher_known_mask = (trainset_unknown[higher_taxa] != '?').all(axis=1)

unknown_species_df = trainset_unknown[species_unknown_mask & higher_known_mask]

print(f"Found {len(unknown_species_df)} rows identified to Genus but missing Species.")

Found 175 rows identified to Genus but missing Species.


In [8]:
unknown_species_df
(unknown_species_df['order'] == "?").any()

False

In [9]:
valid_genera_list = tree_df['order'].unique()
# valid_genera_list = tree_df['genus'].unique()
valid_genera = unknown_species_df[unknown_species_df['order'].isin(valid_genera_list)]
# valid_genera = unknown_species_df[unknown_species_df['genus'].isin(valid_genera_list)]
valid_genera

,phylum,class,order,family
50,Ascomycota,Eurotiomycetes,Chaetothyriales,?
64,Ascomycota,Leotiomycetes,Helotiales,?
76,Ascomycota,Sordariomycetes,Xylariales,?
99,Zoopagomycota,Zoopagomycetes,Zoopagales,?
108,Basidiomycota,Agaricomycetes,Agaricales,?
...,...,...,...,...
2542003,Ascomycota,Lecanoromycetes,Pertusariales,?
3445984,Ascomycota,Eurotiomycetes,Coryneliales,?
3507428,Ascomycota,Sordariomycetes,Melanosporales,?
3990974,Basidiomycota,Tremellomycetes,Holtermanniales,?


In [10]:
# cols = ['phylum', 'class', 'order', 'family', 'genus']

cols = ['phylum', 'class', 'order']

set_unknown = set(map(tuple, valid_genera[cols].values))
set_tree = set(map(tuple, tree_df[cols].values))

is_subset = set_unknown.issubset(set_tree)

print(f"Is every full row in the unknown data present in the tree? {is_subset}")

Is every full row in the unknown data present in the tree? True


In [11]:
missing_paths = set_unknown - set_tree

print(f"Number of unique missing taxonomic paths: {len(missing_paths)}")

for path in list(missing_paths):
    print(path)

Number of unique missing taxonomic paths: 0


In [11]:
valid_genera[valid_genera['genus'] == 'Tetracladium']

,phylum,class,order,family,genus,species
2101,Ascomycota,Leotiomycetes,Helotiales,Helotiaceae,Tetracladium,?
5029865,Ascomycota,Leotiomycetes,Helotiales,Calloriaceae,Tetracladium,?


In [12]:
valid_genera = valid_genera.drop([2101, 5029865])

In [ ]:
cols = ['phylum', 'class', 'order', 'family', 'genus']

set_unknown = set(map(tuple, valid_genera[cols].values))
set_tree = set(map(tuple, tree_df[cols].values))

is_subset = set_unknown.issubset(set_tree)

print(f"Is every full row in the unknown data present in the tree? {is_subset}")

Is every full row in the unknown data present in the tree? True


In [12]:
valid_genera

,phylum,class,order,family
50,Ascomycota,Eurotiomycetes,Chaetothyriales,?
64,Ascomycota,Leotiomycetes,Helotiales,?
76,Ascomycota,Sordariomycetes,Xylariales,?
99,Zoopagomycota,Zoopagomycetes,Zoopagales,?
108,Basidiomycota,Agaricomycetes,Agaricales,?
...,...,...,...,...
2542003,Ascomycota,Lecanoromycetes,Pertusariales,?
3445984,Ascomycota,Eurotiomycetes,Coryneliales,?
3507428,Ascomycota,Sordariomycetes,Melanosporales,?
3990974,Basidiomycota,Tremellomycetes,Holtermanniales,?


In [13]:
valid_genera['species'] = "?"
valid_genera['genus'] = "?"


/tmp/ipykernel_274899/1001858863.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_genera['species'] = "?"
/tmp/ipykernel_274899/1001858863.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_genera['genus'] = "?"


In [14]:
valid_genera

,phylum,class,order,family,species,genus
50,Ascomycota,Eurotiomycetes,Chaetothyriales,?,?,?
64,Ascomycota,Leotiomycetes,Helotiales,?,?,?
76,Ascomycota,Sordariomycetes,Xylariales,?,?,?
99,Zoopagomycota,Zoopagomycetes,Zoopagales,?,?,?
108,Basidiomycota,Agaricomycetes,Agaricales,?,?,?
...,...,...,...,...,...,...
2542003,Ascomycota,Lecanoromycetes,Pertusariales,?,?,?
3445984,Ascomycota,Eurotiomycetes,Coryneliales,?,?,?
3507428,Ascomycota,Sordariomycetes,Melanosporales,?,?,?
3990974,Basidiomycota,Tremellomycetes,Holtermanniales,?,?,?


In [15]:
cols = ["phylum", "class", "order", "family", "genus", "species"]

unknown_from_trainset = trainset.merge(valid_genera[cols], on=cols, how="inner")

In [16]:
cols = ["phylum", "class", "order", "family", "genus", "species"]

check = unknown_from_trainset.merge(
    valid_genera[cols],
    on=cols,
    how="left",
    indicator=True
)

invalid = check[check["_merge"] != "both"]

print(len(invalid))  # should be 0

0


In [33]:
unknown_from_trainset

,phylum,class,order,family,genus,species,dna_barcode
0,Ascomycota,Eurotiomycetes,Chaetothyriales,?,?,?,AGAGGAAGTAAAAGTCGTAACAAGGTTTCCGTAGGTGAACCTGCGG...
1,Ascomycota,Leotiomycetes,Helotiales,?,?,?,CCGAGCTCGTGCCCTTCGGGGTAGACCTCCATCCGTGCCTATCCAA...
2,Zoopagomycota,Zoopagomycetes,Zoopagales,?,?,?,AGCGAAGCCCGATGCGGGTGCAAACTCAATGCTGCGCTAAGCGGCC...
3,Basidiomycota,Agaricomycetes,Agaricales,?,?,?,TTGAATGTTAACATGGGGATGTTGCCGACCAACTTGTCTTGGTATT...
4,Zoopagomycota,Zoopagomycetes,Zoopagales,?,?,?,AGCGAAGCCCGCTGTGCGGGTGCAAAACCAATGCTGCGCAAGCGGC...
...,...,...,...,...,...,...,...
344025,Ascomycota,Sordariomycetes,Sordariales,?,?,?,CAAAAGAGTTGAAAAACTCCCAAAACCCACTGTGAACCTACCTATA...
344026,Basidiomycota,Microbotryomycetes,Microbotryales,?,?,?,GTGAATATTAGGGCGTCTACTTCGGTAGGGCCTGACCTCCACTTTC...
344027,Zoopagomycota,Zoopagomycetes,Zoopagales,?,?,?,ACGAAGCCCGCTGTGCGGGTGTAATACTCAATGCTGCTATTGCGGC...
344028,Ascomycota,Dothideomycetes,Venturiales,?,?,?,ACCGAGTGTGGCCCCATTTTTTATAAAGGGGTACCTCCCGTCCTTT...


In [ ]:
# unknown_from_trainset.to_csv("unknown_from_trainset.csv", index=False)

In [59]:
unknown_from_trainset.describe()

,phylum,class,order,family,genus,species,dna_barcode
count,3483665,3483665,3483665,3483665,3483665,3483665,3483665
unique,13,52,166,526,1956,1,3483665
top,Basidiomycota,Agaricomycetes,Agaricales,Russulaceae,Russula,?,TTGAATAAAATTTGAACAGGTTGTTGCTGGCTCCTCAAGGCATGTG...
freq,1938645,1799356,743654,339843,285310,3483665,1


In [43]:
unknown_from_trainset["order"].value_counts()

order
Helotiales            41153
Zoopagales            25957
Sordariales           25334
Agaricales            25072
Chaetothyriales       23408
                      ...  
Ceraceosorales            1
Pertusariales             1
Melanosporales            1
Holtermanniales           1
Neocallimastigales        1
Name: count, Length: 138, dtype: int64

In [63]:
counts = unknown_from_trainset["order"].value_counts()

# Keep only the orders with 5 or more occurrences
filtered_counts = counts[counts >= 10]
filtered_counts

order
Helotiales            41153
Zoopagales            25957
Sordariales           25334
Agaricales            25072
Chaetothyriales       23408
                      ...  
Spiculogloeales          14
Baeomycetales            14
Gyalectales              11
Peltigerales             10
Lichenostigmatales       10
Name: count, Length: 115, dtype: int64

In [64]:
unknown_from_trainset_filtered = unknown_from_trainset.groupby("order").filter(lambda x: len(x) >= 10)

In [65]:
unknown_from_trainset_filtered

,phylum,class,order,family,genus,species,dna_barcode
0,Ascomycota,Eurotiomycetes,Chaetothyriales,?,?,?,AGAGGAAGTAAAAGTCGTAACAAGGTTTCCGTAGGTGAACCTGCGG...
1,Ascomycota,Leotiomycetes,Helotiales,?,?,?,CCGAGCTCGTGCCCTTCGGGGTAGACCTCCATCCGTGCCTATCCAA...
2,Zoopagomycota,Zoopagomycetes,Zoopagales,?,?,?,AGCGAAGCCCGATGCGGGTGCAAACTCAATGCTGCGCTAAGCGGCC...
3,Basidiomycota,Agaricomycetes,Agaricales,?,?,?,TTGAATGTTAACATGGGGATGTTGCCGACCAACTTGTCTTGGTATT...
4,Zoopagomycota,Zoopagomycetes,Zoopagales,?,?,?,AGCGAAGCCCGCTGTGCGGGTGCAAAACCAATGCTGCGCAAGCGGC...
...,...,...,...,...,...,...,...
344025,Ascomycota,Sordariomycetes,Sordariales,?,?,?,CAAAAGAGTTGAAAAACTCCCAAAACCCACTGTGAACCTACCTATA...
344026,Basidiomycota,Microbotryomycetes,Microbotryales,?,?,?,GTGAATATTAGGGCGTCTACTTCGGTAGGGCCTGACCTCCACTTTC...
344027,Zoopagomycota,Zoopagomycetes,Zoopagales,?,?,?,ACGAAGCCCGCTGTGCGGGTGTAATACTCAATGCTGCTATTGCGGC...
344028,Ascomycota,Dothideomycetes,Venturiales,?,?,?,ACCGAGTGTGGCCCCATTTTTTATAAAGGGGTACCTCCCGTCCTTT...


In [66]:
train_df = unknown_from_trainset_filtered.sample(n=17735, random_state=42)

In [19]:
del trainset
del tree_df

In [67]:
# 2. Get the remaining sequences (to ensure no overlap)
remaining_df = unknown_from_trainset_filtered.drop(train_df.index)

In [68]:
remaining_df

,phylum,class,order,family,genus,species,dna_barcode
0,Ascomycota,Eurotiomycetes,Chaetothyriales,?,?,?,AGAGGAAGTAAAAGTCGTAACAAGGTTTCCGTAGGTGAACCTGCGG...
1,Ascomycota,Leotiomycetes,Helotiales,?,?,?,CCGAGCTCGTGCCCTTCGGGGTAGACCTCCATCCGTGCCTATCCAA...
3,Basidiomycota,Agaricomycetes,Agaricales,?,?,?,TTGAATGTTAACATGGGGATGTTGCCGACCAACTTGTCTTGGTATT...
4,Zoopagomycota,Zoopagomycetes,Zoopagales,?,?,?,AGCGAAGCCCGCTGTGCGGGTGCAAAACCAATGCTGCGCAAGCGGC...
5,Ascomycota,Sordariomycetes,Annulatascales,?,?,?,CAGAGTTGAAGAACTCCCAAACCCACTGTGAATCTACCTGTACTTG...
...,...,...,...,...,...,...,...
344024,Mortierellomycota,Mortierellomycetes,Mortierellales,?,?,?,TAATAGAGTAAATTTTATGGTGCTCTAAAAAAATCCCATATCCACC...
344025,Ascomycota,Sordariomycetes,Sordariales,?,?,?,CAAAAGAGTTGAAAAACTCCCAAAACCCACTGTGAACCTACCTATA...
344026,Basidiomycota,Microbotryomycetes,Microbotryales,?,?,?,GTGAATATTAGGGCGTCTACTTCGGTAGGGCCTGACCTCCACTTTC...
344028,Ascomycota,Dothideomycetes,Venturiales,?,?,?,ACCGAGTGTGGCCCCATTTTTTATAAAGGGGTACCTCCCGTCCTTT...


In [69]:
# grouped_remaining = remaining_df.groupby('genus')
grouped_remaining = remaining_df.groupby('order')

def pick_test_sequence(taxon):
    try:
        # Get all available sequences for this genus that aren't in train
        available = grouped_remaining.get_group(taxon)
        # Sample 1 (randomly) from what's left
        return available.sample(n=1)
    except (KeyError, ValueError):
        # Handle cases where a genus might not have a second sequence
        return None

test_list = []
for taxon in train_df['order']:
    sibling = pick_test_sequence(taxon)
    if sibling is not None:
        test_list.append(sibling)


test_df = pd.concat(test_list).reset_index(drop=True)
train_df = train_df.reset_index(drop=True)

In [23]:
def check_overlap(df_small, df_big):
    df_small = df_small[["dna_barcode"]]
    df_big = df_big[["dna_barcode"]]

    check_merge = df_small.merge(df_big, how='left', indicator=True)
    all_unique = (check_merge['_merge'] == 'left_only').all()
    print(f"Zero overlap (Every row is unseen): {all_unique}")

def check_merge(df_small, df_big):
    df_small = df_small.drop(columns=["dna_barcode"]).drop_duplicates()
    df_big = df_big.drop(columns=["dna_barcode"]).drop_duplicates()

    check_merge = df_small.merge(df_big, how='left', indicator=True)
    is_subset = (check_merge['_merge'] == 'both').all()
    print(f"Every row exists in the other: {is_subset}")

In [70]:
check_overlap(test_df, train_df)
check_merge(test_df.drop(columns=["species", "genus", "family"]), train_df.drop(columns=["species", "genus", "family"]))

Zero overlap (Every row is unseen): True
Every row exists in the other: True


In [71]:
train_df.to_csv("train_unknown.csv", index=False)
test_df.to_csv("test_unknown.csv", index=False)

In [ ]:
# unknown_from_trainset = unknown_from_trainset.sample(n=5000, random_state=42)

In [ ]:
# unknown_from_trainset = standardize_barcode(unknown_from_trainset)
# unknown_from_trainset.to_csv("unknown_species.csv", index=False)

## Creating Targets_csv

In [79]:
# unknown_train_df = pd.read_csv("../datasets/mycoai_full/train_unknown.csv")
# unknown_test_df = pd.read_csv("../datasets/mycoai_full/test_unknown.csv")

unknown_from_trainset = pd.read_csv("test_unknown.csv")

taxonomy = np.load('../datasets/mycoai_full/taxonomy.npz', allow_pickle=True)
name_to_id_map = taxonomy["name_to_id_map"].item()
parents = taxonomy["parents"]
unk = taxonomy["unk"]
indices = np.where(unk)[0]
parent_to_idx = dict(zip(parents[indices].tolist(), indices.tolist()))

In [80]:
i = 0
for _, row in unknown_from_trainset.iterrows():
    parent_name = row["order"]
    parent_id = name_to_id_map[parent_name]
    child_id = parent_to_idx[parent_id]
    unknown_from_trainset.at[_, "NodeID"] = child_id
    i += 1

unknown_from_trainset['NodeID'] = unknown_from_trainset['NodeID'].astype(int)

In [81]:
unknown_from_trainset

,phylum,class,order,family,genus,species,dna_barcode,NodeID
0,Ascomycota,Eurotiomycetes,Onygenales,?,?,?,ATCAGCCGTGAAGGTCGCCCTGAAAAGGGAATGACCTCTCACCTGG...,477
1,Ascomycota,Eurotiomycetes,Chaetothyriales,?,?,?,CCGAGTAAAAGGGTTTAACAGCCCGCACTCCAACCCTATGTGTATC...,459
2,Ascomycota,Eurotiomycetes,Chaetothyriales,?,?,?,CCGAGTTAGGGTCTTCATGGCCTGACCTCCAACCCTATGTCTACTA...,459
3,Ascomycota,Leotiomycetes,Helotiales,?,?,?,CAGAGAACATCGCCCTCACGGGTGACTCTCCAACCCTATGTTATTA...,573
4,Chytridiomycota,Spizellomycetes,Spizellomycetales,?,?,?,AAAAAAATCCGTGGCGAGCACCGTCTCATGCACGTTACATGAATGC...,1065
...,...,...,...,...,...,...,...,...
17730,Blastocladiomycota,Blastocladiomycetes,Blastocladiales,?,?,?,TGAATATTGACGAGTCGGTTGCTCTCTTTTTTTTTATTAAAGGGGG...,1052
17731,Basidiomycota,Agaricomycetes,Boletales,?,?,?,TCGAAATCTGAGTAGGAAGACGGAAGGTGAAAAGGACTGTCGCCTC...,846
17732,Basidiomycota,Agaricomycetes,Atheliales,?,?,?,TTGAATTACGGGCGAGGGTTGTCGCTGGCCTCTCGGGGCATGTGCA...,840
17733,Ascomycota,Dothideomycetes,Venturiales,?,?,?,ACCGAGTGCGAACCCCCCCCAATAAAAAAGGGGGGTTACCTCCCGT...,455


In [82]:
parent_name = "Onygenales"
unk_id = 477

print(taxonomy["names"][unk_id] == 'unk')
print(name_to_id_map[parent_name] == parents[unk_id])

True
True


In [77]:
# def get_list_of_targets(sequences_df, hierarchy, name_to_id_map):
#     # create training-targets
#     sequences_df.reset_index(drop=True, inplace=True)
#     list_of_targets = []
#     for _, row in sequences_df.iterrows():
#         row_ids = [-1] * 7
#         row_ids[0] = 1
#         for i, rank in enumerate(hierarchy):
#             row_ids[i+1] = name_to_id_map.get(row[rank], -1)

#         list_of_targets.append(row_ids)

#     return list_of_targets

def get_list_of_targets_unk(sequences_df, hierarchy, name_to_id_map):
    # create training-targets
    sequences_df.reset_index(drop=True, inplace=True)
    list_of_targets = []
    for _, row in sequences_df.iterrows():
        row_ids = [-1] * 7
        row_ids[0] = 1
        for i, rank in enumerate(hierarchy):
            if rank == "family":
                row_ids[i+1] = row["NodeID"]
            else:
                row_ids[i+1] = name_to_id_map.get(row[rank], -1)

        list_of_targets.append(row_ids)

    return list_of_targets

def save_test_labels(list_of_targets, name):
    targets_df = pd.DataFrame(list_of_targets)
    taxa_levels = ['kingdom_id', 'phylum_id', 'class_id', 'order_id', 'family_id', 'genus_id', 'species_id']
    targets_df.columns = taxa_levels
    targets_df.insert(0, 'root_id', 0)

    targets_df.to_csv(f'{name}.csv', index=False)

    print("Test labels saved.")

    return

def save_training_targets(list_of_targets, name):

    targets_df = pd.DataFrame(list_of_targets)
    targets_df_tr = targets_df.T
    targets_df_tr.to_csv(f'{name}.csv', index=True)

    print("Training targets saved.")

    return

hierarchy = ["phylum", "class", "order", "family", "genus", "species"]

In [83]:
list_of_targets = get_list_of_targets_unk(unknown_from_trainset, hierarchy, name_to_id_map)
save_test_labels(list_of_targets, "unknown_labels")
save_training_targets(list_of_targets, "unknown_train-targets")

Test labels saved.
Training targets saved.


In [84]:
df_1 = pd.read_csv("family_labels.csv")
df_2 = pd.read_csv("unknown_labels.csv")
pd.testing.assert_frame_equal(df_1, df_2)
df_1.equals(df_2)

True

In [ ]:
# missing_species = pd.read_csv("missing_species.csv")
# list_of_targets = get_list_of_targets(missing_species, hierarchy, name_to_id_map)
# save_test_labels(list_of_targets, "missing_labels")
# save_training_targets(list_of_targets, "missing_train-targets")

Test labels saved.
Training targets saved.


## Concatenation

In [106]:
df1 = pd.read_csv("../datasets/mycoai_full/train_labels.csv")
df2 = pd.read_csv("../datasets/mycoai_full/unknown_data/unknown_labels.csv")
# df3 = pd.read_csv("../datasets/mycoai_full/unknown_data/separate_levels/family_test.csv")

combined = pd.concat([df1, df2], axis=0, ignore_index=True)
combined.to_csv("unknown.csv", index=False)

In [110]:
df1 = pd.read_csv("../datasets/mycoai_full/train-targets.csv").iloc[:, 1:]
df2 = pd.read_csv("../datasets/mycoai_full/unknown_data/unknown_train-targets.csv").iloc[:, 1:]
# df3 = pd.read_csv("../datasets/mycoai_full/unknown_data/family_train-targets.csv").iloc[:, 1:]

combined = pd.concat([df1.T, df2.T], axis=0, ignore_index=True)
combined = combined.T
combined.to_csv("unknown_train-targets.csv", index=True)

In [5]:
train = np.load("../datasets/mycoai_full/mamba/train_embeddings.npz")["embeddings"]
extra = np.load("mamba_unknown_train.npz")["embeddings"]
concatenated = np.vstack([train, extra])
np.savez_compressed("../datasets/mycoai_full/mamba/unknown/train_embeddings.npz", embeddings=np.asarray(concatenated))

In [ ]:
bert = np.load("../datasets/mycoai_hybrid/bert_base_embeddings.npz")["embeddings"]
mamba = np.load("../datasets/mycoai_hybrid/base_embeddings.npz")["embeddings"]
np.savez_compressed("hybrid_embeddings.npz", bert=np.asarray(bert), mamba=np.asarray(mamba))

In [5]:
bert = np.load("../datasets/mycoai_hybrid/train_embeddings.npz")["bert"]
mamba = np.load("../datasets/mycoai_hybrid/train_embeddings.npz")["mamba"]

# Bioscan Preprocessing

In [40]:
import sys
from pathlib import Path

# Prepend so this repo's `protax` / `scripts` win over site-packages in the venv.
_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "protax").is_dir() and (_p / "scripts").is_dir():
        project_root = str(_p.resolve())
        break
else:
    project_root = str((_here / "..").resolve())

sys.path.insert(0, project_root)

from scripts.bioscan_convert import *

In [54]:
data_dir = "../datasets/mycoai"
hierarchy = ["phylum", "class", "order", "family", "genus", "species"]
# full_df = pd.read_csv(f"{data_dir}/test1_clean.csv")
# convert_to_taxonomy(full_df, hierarchy, data_dir)    

train_sequences_df = pd.read_csv(f"{data_dir}/trainset_clean.csv")
targets_df = train_sequences_df.copy()

# assign_sequences_to_taxonomy(train_sequences_df, hierarchy, data_dir)

list_of_targets = get_list_of_targets(targets_df, hierarchy, data_dir)
# save_training_targets(list_of_targets, data_dir)
# save_test_labels(list_of_targets, data_dir)

## Debugging

In [48]:
hierarchy = ["phylum", "class", "order", "family", "genus", "species"]
df = df[hierarchy + ["dna_barcode"]]

In [ ]:
taxon_to_prior_map, unknown_prior_per_rank = assign_priors(df, hierarchy)
max_seq_length = df['dna_barcode'].str.len().max()


df = df.drop(columns=['dna_barcode'])
df = df.drop_duplicates().reset_index(drop=True)
taxon_to_id_map = assign_global_ids(df, hierarchy)

df = add_NodeID_and_lvl(df, taxon_to_id_map, hierarchy)
df = add_ParentID(df, taxon_to_id_map, hierarchy)
df =build_taxonomy_dataframe(df, taxon_to_id_map, taxon_to_prior_map, hierarchy)


### Adding Unk Length Mismatch Problem

In [ ]:
# adding unknown nodes function

# nid = df.index.max() + 1
# parents = df["pid"].unique() 
# parents = parents[2:] # to avoid adding unknown nodes to kingdom and above (should it be 1 or 2)????
# parent_levels = df.loc[parents, "lvl"]


# lvl_to_rank = {i + 2: rank for i, rank in enumerate(hierarchy)}
# unk_priors = [
#     unknown_prior_per_rank.get(lvl_to_rank.get(lvl + 1), 0) 
#     for lvl in parent_levels
# ]


# len(unk_priors)
# unk_df = pd.DataFrame({
#     "nid": range(nid, nid + len(parents)),
#     "pid": parents,
#     "lvl": parent_levels + 1,
#     "name": "unk",
#     "prior": unk_priors
# }).set_index("nid")


3406

In [ ]:
# df.set_index("nid", inplace=True)
# df = add_unknown_nodes(df, hierarchy, unknown_prior_per_rank)      
# df

# Convert to .aln

In [1]:
def remove_question_marks(df):
    initial_count = len(df)
    rows_with_question_mark = df.isin(['?']).any(axis=1)
    df = df[~rows_with_question_mark]
    removed_count = initial_count - len(df)
    print(f"Removed {removed_count} rows containing '?'")

    return df

def mycoai_fasta_to_df(file_path):
    # Define the expected levels in the taxonomy
    levels = {'p__': 'phylum', 'c__': 'class', 
              'o__': 'order', 'f__': 'family', 'g__': 'genus', 's__': 'species'}
    
    data = []
    for record in SeqIO.parse(file_path, "fasta"):
        # prefill with NA
        row = {name: pd.NA for name in levels.values()}
        
        # Split header: Accession|Taxonomy|ID
        parts = record.description.split('|')
        row['accession'] = parts[0]
        row['dna_barcode'] = str(record.seq)
        
        # Parse taxonomy string if it exists
        if len(parts) > 1:
            taxa_parts = parts[1].split(';')
            for item in taxa_parts:
                prefix = item[:3] # Get 'k__', 'p__', etc.
                if prefix in levels:
                    row[levels[prefix]] = item[3:].replace('_', ' ')
        
        data.append(row)

    hierarchy = ["phylum", "class", "order", "family", "genus", "species"]

    df = pd.DataFrame(data)
    # df = remove_question_marks(df)
    
    return df, hierarchy

def save_df_to_protax_format(df):
    # The ranks you want to include in the header
    hierarchy = ['phylum', 'class', 'order', 'family', 'genus', 'species']
    
    with open("refs.aln", 'w', encoding='latin-1') as f:
        for _, row in df.iterrows():
            taxonomy_str = ",".join([str(row[rank]) for rank in hierarchy if pd.notna(row[rank])])            
            header = f"{taxonomy_str}\t\n"            
            sequence = str(row['dna_barcode']).strip()
            
            f.write(header)
            f.write(f"{sequence}\n")

In [ ]:
# train_df, hierarchy = mycoai_fasta_to_df("../datasets/mycoai/test3.fasta")
df = pd.read_csv("unknown_test.csv")
hierarchy = ["phylum", "class", "order", "family", "genus", "species"]
save_df_to_protax_format(df)

# 350K Data Study

In [ ]:
def sequences_insights(df):
    sequence_counts = df['dna_barcode'].value_counts()
    duplicated_sequences = sequence_counts[sequence_counts > 1]
    print(f"Total Unique Sequences: {len(sequence_counts)}")
    print(f"Number of sequences that appear more than once: {len(duplicated_sequences)}")

def taxonomic_paths_insights(df):
    taxa_cols = ['phylum', 'class', 'order', 'family', 'genus', 'species']
    path_counts = df.groupby(taxa_cols).size()
    print(f"Total Unique Taxonomic Paths: {len(path_counts)}")

    only_1 = (path_counts == 1).sum()
    only_2 = (path_counts == 2).sum()
    only_3 = (path_counts == 3).sum()
    only_4 = (path_counts == 4).sum()
    only_5 = (path_counts == 5).sum()
    more_than_5 = (path_counts > 5).sum()
    print(f"Taxonomic paths with exactly 1 sequence: {only_1}")
    print(f"Taxonomic paths with exactly 2 sequences: {only_2}")
    print(f"Taxonomic paths with exactly 3 sequences: {only_3}")
    print(f"Taxonomic paths with exactly 4 sequences: {only_4}")
    print(f"Taxonomic paths with exactly 5 sequences: {only_5}")
    print(f"Taxonomic paths with more than 5 sequences: {more_than_5}")

def retrieve_barcode_conflicts(df):
    genus_counts = df.groupby('dna_barcode')['genus'].nunique()
    family_counts = df.groupby('dna_barcode')['family'].nunique()
    order_counts = df.groupby('dna_barcode')['order'].nunique()
    class_counts = df.groupby('dna_barcode')['class'].nunique()
    phylum_counts = df.groupby('dna_barcode')['phylum'].nunique()

    bad_barcodes_genus = genus_counts[genus_counts > 1].index
    bad_barcodes_family = family_counts[family_counts > 1].index
    bad_barcodes_order = order_counts[order_counts > 1].index
    bad_barcodes_class = class_counts[class_counts > 1].index
    bad_barcodes_phylum = phylum_counts[phylum_counts > 1].index

    diff_genus_df = trainset[trainset['dna_barcode'].isin(bad_barcodes_genus)].sort_values('dna_barcode')
    diff_family_df = trainset[trainset['dna_barcode'].isin(bad_barcodes_family)].sort_values('dna_barcode')
    diff_order_df = trainset[trainset['dna_barcode'].isin(bad_barcodes_order)].sort_values('dna_barcode')
    diff_class_df = trainset[trainset['dna_barcode'].isin(bad_barcodes_class)].sort_values('dna_barcode')
    diff_phylum_df = trainset[trainset['dna_barcode'].isin(bad_barcodes_phylum)].sort_values('dna_barcode')

    print(f"Number of bad barcodes under phyla: {len(diff_phylum_df)}")
    print(f"Number of bad barcodes under classes: {len(diff_class_df)}")
    print(f"Number of bad barcodes under orders: {len(diff_order_df)}")
    print(f"Number of bad barcodes under families: {len(diff_family_df)}")
    print(f"Number of bad barcodes under genera: {len(diff_genus_df)}")

    return diff_phylum_df, diff_class_df, diff_order_df, diff_family_df, diff_genus_df

def find_barcode_conflicts(df):
    conflicts = df.groupby('dna_barcode')['species'].nunique()
    shared_barcodes = conflicts[conflicts > 1].index
    print(f"Number of sequences shared by multiple taxa: {len(shared_barcodes)}")

    conflict_df = df[df['dna_barcode'].isin(shared_barcodes)]
    conflict_df = conflict_df.sort_values('dna_barcode')
    print(f"Total rows with shared barcodes: {len(conflict_df)}")

    return conflict_df

def gain_insights(df):
    sequences_insights(df)
    print("-"*100)
    taxonomic_paths_insights(df)
    print("-"*100)
    conflict_df = find_barcode_conflicts(df)
    print("-"*100)
    diff_phylum_df, diff_class_df, diff_order_df, diff_family_df, diff_genus_df = retrieve_barcode_conflicts(df)
    return conflict_df, diff_phylum_df, diff_class_df, diff_order_df, diff_family_df, diff_genus_df


## Pre-train Cleaned

In [15]:
trainset = pd.read_csv("../datasets/mycoai_full/trainset_clean.csv")
trainset.nunique()

phylum             13
class              56
order             182
family            606
genus            2549
species         14374
dna_barcode    350838
dtype: int64

In [19]:
conflict_df, diff_phylum_df, diff_class_df, diff_order_df, diff_family_df, diff_genus_df = gain_insights(trainset)

Total Unique Sequences: 350838
Number of sequences that appear more than once: 3294
----------------------------------------------------------------------------------------------------
Total Unique Taxonomic Paths: 14374
Taxonomic paths with exactly 1 sequence: 0
Taxonomic paths with exactly 2 sequences: 0
Taxonomic paths with exactly 3 sequences: 16
Taxonomic paths with exactly 4 sequences: 2619
Taxonomic paths with exactly 5 sequences: 1770
Taxonomic paths with more than 5 sequences: 9969
----------------------------------------------------------------------------------------------------
Number of sequences shared by multiple species: 3294
Total rows with shared barcodes: 7151
----------------------------------------------------------------------------------------------------
Number of bad barcodes under phyla: 28
Number of bad barcodes under classes: 158
Number of bad barcodes under orders: 294
Number of bad barcodes under families: 418
Number of bad barcodes under genera: 1047


## Benchmark Cleaned

In [26]:
benchmark_df = pd.read_csv("../datasets/mycoai_full/benchmark_cleaned.csv")
merged_df = benchmark_df.merge(trainset, how='inner')
is_subset = len(merged_df) == len(benchmark_df)
print(f"Is benchmarkf a subset of trainset? {is_subset}")

Is benchmarkf a subset of trainset? True


In [2]:
train = pd.read_csv("../datasets/mycoai_full/trainset.csv")
test = pd.read_csv("../datasets/mycoai_full/test3.csv")

# Appendix

In [44]:
# Open the file and count lines one by one
with open('../datasets/mycoai/list_of_labels_nodeIDs.txt', 'r') as f:
    line_count = sum(1 for line in f)

print(f"Total lines: {line_count}")

Total lines: 50000


In [45]:
import pandas as pd

df = pd.read_csv('../datasets/mycoai/train-targets.csv')

# Returns (rows, columns)
print(df.shape)

# If you only want the row count
print(len(df))

(7, 50001)
7


In [ ]:
taxonomy = np.load('../datasets/mycoai/full_tax/taxonomy.npz', allow_pickle=True)
print("Arrays in this file:", taxonomy.files)

print(taxonomy['segments'].shape)
print(taxonomy['refs'].shape)

taxonomy['ranks']

array([0, 1, 2, ..., 7, 7, 7])

In [ ]:
# taxonomy = np.load('../datasets/fin_protax_37k/taxonomy.npz', allow_pickle=True)
# print("Arrays in this file:", taxonomy.files)
# taxonomy['max_seq_length']

# data_dict = {key: taxonomy[key] for key in taxonomy.files}
# data_dict['max_seq_length'] = 658
# np.savez('../datasets/fin_protax_37k/taxonomy.npz', **data_dict)

Arrays in this file: ['segments', 'unk', 'paths', 'refs', 'ok_pos', 'node_state', 'parents', 'prior', 'ranks', 'n2s_indices', 'n2s_indptr', 'max_seq_length']


array(658)

In [61]:
import jax.numpy as jnp
import numpy as np

data = np.load("../datasets/mycoai/test_embeddings.npz")
print("Arrays in file:", data.files)
jax_array = jnp.array(data['embeddings'])

print("Array shape:", jax_array.shape)

Arrays in file: ['embeddings']
Array shape: (50000, 768)


In [ ]:
def convert_train_targets_to_labels(targets_df):
    taxa_levels = ['kingdom_id', 'phylum_id', 'class_id', 'order_id', 'family_id', 'genus_id', 'species_id']
    targets_df = targets_df.T
    targets_df = targets_df.iloc[1:].reset_index(drop=True)
    targets_df.columns = taxa_levels
    targets_df.insert(0, 'root_id', 0)

    targets_df.to_csv('test_labels.csv', index=False)

df = pd.read_csv('../datasets/fin_protax_37k/train-targets.csv')
convert_train_targets_to_labels(df)

,Unnamed: 0,0,1,2,3,4,5,6,7,8,...,37412,37413,37414,37415,37416,37417,37418,37419,37420,37421
0,0,7,7,7,7,7,7,7,7,7,...,7,7,7,7,7,7,7,7,2,7
1,1,44,56,56,63,63,46,46,46,46,...,63,56,56,46,46,46,46,46,19,46
2,2,404,797,797,862,862,490,490,490,490,...,855,798,798,532,532,532,532,532,232,518
3,3,1647,2853,2853,2986,2986,1936,1936,1936,1936,...,2968,2855,2855,2057,2057,2057,2057,2057,1232,2012
4,4,4106,6233,6233,6391,6391,4642,4642,4642,4642,...,6364,6235,6235,4866,4866,4866,4866,4866,3411,4752
5,5,8689,14870,14870,15221,15221,10194,10194,10196,10196,...,15160,14887,14887,10966,10966,10966,10966,10966,7229,10502
6,6,-1,-1,-1,-1,-1,-1,-1,-1,-1,...,48091,47322,47322,32018,32018,32022,32022,32022,17476,-1


### Checking targets

In [7]:
# # code to concatenate test_labels
# path1 = "../datasets/mycoai/full_tax/test_labels.csv"
# path2 = "ood_data/unknown_labels.csv"

# df1 = pd.read_csv(path1)
# df2 = pd.read_csv(path2)

# combined = pd.concat([df1, df2], axis=0, ignore_index=True)
# combined.to_csv("test_labels.csv", index=False)
# combined = pd.read_csv("test_labels.csv")
# combined

In [8]:
# # code to concatenate train-targets
# path1 = "../datasets/mycoai/full_tax/train-targets.csv"
# path2 = "ood_data/unknown_train-targets.csv"

# df1 = pd.read_csv(path1).iloc[:, 1:]
# df2 = pd.read_csv(path2).iloc[:, 1:]

# combined = pd.concat([df1.T, df2.T], axis=0, ignore_index=True)
# combined = combined.T
# combined.to_csv("train-targets.csv", index=True)
# combined = pd.read_csv("train-targets.csv")
# combined

In [9]:
# Cut embeddings

# embeddings = np.load("../datasets/mycoai/full_tax/unknown_and_missing/test_embeddings.npz")["embeddings"]
# embeddings = embeddings[:-3607, :]
# np.savez_compressed("test_embeddings.npz", embeddings=embeddings)